### RAG Pipline- Data Ingestion to Vector DB Pipeline

In [3]:
import os
from importlib.metadata import metadata

from jedi.plugins.stdlib import collections_namedtuple
from langchain_classic import text_splitter
from langchain_classic.schema import embeddings
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

from pydantic_settings.sources.providers import dotenv
from sqlalchemy.ext.asyncio import result

/tmp/ipykernel_16335/1090164722.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader


In [4]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: s1-cybersecurity-essentials-interview-guide.pdf
  ✓ Loaded 10 pages

Processing: 2023-Wireshark-workshop-slides.pdf
  ✗ Error: Cannot read an empty file

Total documents loaded: 10


In [5]:
### Text splitting - get into chunks

def split_documents(documents,chunk_size=1000, chunk_overlap=200):
    """Split document into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    #Show example of a chunk
    if split_docs:
        print(f"\nExample chunk: ")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [12]:
# Check if the variable exists and has content
try:
    print(f"Type: {type(all_pdf_documents)}")
    print(f"Total documents loaded: {len(all_pdf_documents)}")
    if len(all_pdf_documents) > 0:
        print("✓ all_pdf_documents is ready!")
    else:
        print("⚠ The list is empty! Check if your PDFs are actually inside '../data'.")
except NameError:
    print("❌ all_pdf_documents is NOT defined yet. Run your loading cell again.")

Type: <class 'list'>
Total documents loaded: 10
✓ all_pdf_documents is ready!


In [9]:
chunks = split_documents(all_pdf_documents)
chunks

split 10 documents into 35 chunks

Example chunk: 
Content: Generated from Notebook Sources | S1 Semester Master Blueprint
 CYBERSECURITY ESSENTIALS & SYSTEM
 PROGRAMMING
 S1 Semester Unit Complete Blueprint & Technical Interview Masterclass
Attribute
Specific...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-04T12:50:00+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-04T12:50:00+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/text_files/s1-cybersecurity-essentials-interview-guide.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 's1-cybersecurity-essentials-interview-guide.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-04T12:50:00+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-04T12:50:00+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/text_files/s1-cybersecurity-essentials-interview-guide.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 's1-cybersecurity-essentials-interview-guide.pdf', 'file_type': 'pdf'}, page_content='Generated from Notebook Sources | S1 Semester Master Blueprint\n CYBERSECURITY ESSENTIALS & SYSTEM\n PROGRAMMING\n S1 Semester Unit Complete Blueprint & Technical Interview Masterclass\nAttribute\nSpecification\nAttribute\nSpecification\nCourse Unit:\nS1 Cybersecurity Essentials & System\nProgramming\nTarget Level:\nInterview-Ready Security Engineer /\nSystems Analyst\nMapped COs:\nCO1 (Threat Modeling), CO2 (Attacks &\nMitigation), CO4 (Risk Analysis), CO5\n(Policie

### embedding And vectorStoreDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/home/s4ge/Documents/CyberProjects/YTRAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedded model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        """
        Generating embedding for a list of texts
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embedded for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

### Initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedded model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6067.69it/s]


Model loaded successfully. Embedding dimension: 384


### Vector Store

In [9]:
import os

class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [15]:
# Run this cell explicitly to create the 'chunks' variable
chunks = split_documents(all_pdf_documents)
print(f"Successfully created {len(chunks)} text chunks!")
chunks

split 10 documents into 35 chunks

Example chunk: 
Content: Generated from Notebook Sources | S1 Semester Master Blueprint
 CYBERSECURITY ESSENTIALS & SYSTEM
 PROGRAMMING
 S1 Semester Unit Complete Blueprint & Technical Interview Masterclass
Attribute
Specific...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-04T12:50:00+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-04T12:50:00+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/text_files/s1-cybersecurity-essentials-interview-guide.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 's1-cybersecurity-essentials-interview-guide.pdf', 'file_type': 'pdf'}
Successfully created 35 text chunks!


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-04T12:50:00+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-04T12:50:00+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/text_files/s1-cybersecurity-essentials-interview-guide.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 's1-cybersecurity-essentials-interview-guide.pdf', 'file_type': 'pdf'}, page_content='Generated from Notebook Sources | S1 Semester Master Blueprint\n CYBERSECURITY ESSENTIALS & SYSTEM\n PROGRAMMING\n S1 Semester Unit Complete Blueprint & Technical Interview Masterclass\nAttribute\nSpecification\nAttribute\nSpecification\nCourse Unit:\nS1 Cybersecurity Essentials & System\nProgramming\nTarget Level:\nInterview-Ready Security Engineer /\nSystems Analyst\nMapped COs:\nCO1 (Threat Modeling), CO2 (Attacks &\nMitigation), CO4 (Risk Analysis), CO5\n(Policie

In [17]:
texts=[doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embedding(texts)

vectorstore.add_documents(chunks, embeddings)

Generating embedded for 35 texts...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.98it/s]

Generated embeddings with shape: (35, 384)
Adding 35 documents to vector store...
Successfully added 35 documents to vector store
Total documents in collection: 35


In [23]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [34]:
rag_retriever.retrieve("RTCP 3-Way Handshake")

Retrieving documents for query: 'RTCP 3-Way Handshake'
Top K: 5, Score threshold: 0.0
Generating embedded for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 88.65it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_3d6ff620_12',
  'content': '4. TCP 3-Way Handshake: Client establishes a reliable TCP connection with the target web server (SYN →\n SYN-ACK → ACK).\n5. HTTP GET & Response: Client issues encrypted HTTPS/TLS or HTTP GET request; server responds with web\n page HTML payload.\n2.2 Transport Layer Mechanics: TCP 3-Way Handshake vs. UDP\nTCP (Transmission Control Protocol) provides connection-oriented, ordered, reliable byte-stream delivery.\nConnection state is established via the 3-Way Handshake:\nHandshake Step\nDirection\nTechnical Exchange & Sequence Mechanics\nStep 1: SYN\nClient → Server\nClient sends SYN packet with an Initial Sequence Number (ISN_C). State:\nSYN_SENT.\nStep 2: SYN-ACK\nServer → Client\nServer responds with SYN-ACK, acknowledging ISN_C + 1 and sending its own\nISN_S. State: SYN_RCVD.\nStep 3: ACK\nClient → Server\nClient acknowledges ISN_S + 1. Connection established on both sides. State:\nESTABLISHED.\n2.3 Core Network Attacks & Technical Defenses\nAt

### Integration Vectordb Context pipeline With LLM output

In [41]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM using an active free-tier model string
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model="openai/gpt-oss-20b",  # Updated to an active model id
    temperature=0.1,
    max_tokens=1024
)

## Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    # Retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    # Generate the answer using Groq LLM
    prompt = f"""Use the following context to answer the question concisely.
    Context:
    {context}

    Question: {query}

    Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [42]:
answer = rag_simple("What is CIA Triad?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is CIA Triad?'
Top K: 3, Score threshold: 0.0
Generating embedded for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 107.95it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


The CIA Triad is the three foundational principles of information security: **Confidentiality, Integrity, and Availability**.


In [43]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is RBAC2?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is RBAC2?'
Top K: 3, Score threshold: 0.1
Generating embedded for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 73.68it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


Answer: **RBAC2** is the RBAC model that extends RBAC1 by adding *constraints*—specifically Static Separation of Duties (SSD) and Dynamic Separation of Duties (DSD).
Sources: [{'source': 's1-cybersecurity-essentials-interview-guide.pdf', 'page': 2, 'score': 0.11000853776931763, 'preview': '\x7f Simple Integrity Property: No Read Down (NRD) — Subject cannot read objects of lower integrity (I(S) ≤ I(O)).\n\x7f Integrity *-Property: No Write Up (NWU) — Subject cannot write to objects of higher integrity (I(S) ≥ I(O)).\nDeep Dive: The NIST RBAC Reference Family (RBAC0 – RBAC3)\nRBAC Model Level\nFu...'}]
Confidence: 0.11000853776931763
Context Preview:  Simple Integrity Property: No Read Down (NRD) — Subject cannot read objects of lower integrity (I(S) ≤ I(O)).
 Integrity *-Property: No Write Up (NWU) — Subject cannot write to objects of higher integrity (I(S) ≥ I(O)).
Deep Dive: The NIST RBAC Reference Family (RBAC0 – RBAC3)
RBAC Model Level
Fu


In [44]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("How many Risk parameters are there ?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'How many Risk parameters are there ?'
Top K: 3, Score threshold: 0.1
Generating embedded for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 90.89it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
underlying read-only physical page, corrupting /etc/passwd to gain root.
MODULE 4: Risk Management, Security Controls & Governance
4.1 Risk Assessment Frameworks & Analysis Mechanics
Risk management identifies, evaluates, and mitigates threats to orga

nizational assets. Based on NIST SP 800-30 /
SP 800-53:
The Core Risk Equation: Risk = Threat × Vulnerability × Impact
Risk Parameter
Formal Definition & Governance Scope
Threat
Any potential cause of an unwanted incident that may result in harm to a system or organization.
Vulnerability
A weakness in an asset or control that could be exploited by one or more threats.
Impact
Magnitude of loss or harm resulting from the consequences of an adversary exploiting a
vulnerability.
Likelihood
Probability that a given threat will successfully exercise a specific vulnerability.
Residual Risk
Remaining risk after security controls and risk treatments have been applied (Residual = Inherent -
Control Efficacy).
Risk Treatment Strategies:

Question: How many Risk parameters are there ?

Answer:

Final Answer: 5

Citations:
[1] s1-cybersecurity-essentials-interview-guide.pdf (page 6)
Summary: The answer is the number five. It represents a single digit value.
History: {'question': 'How many Risk para